# 01 — Data Preprocessing
Load, clean, reshape and save the Ausgrid solar dataset.

## 1. Imports & paths

In [5]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")

RAW_DIR  = "D:/Minor Project/raw_data"
PROC_DIR = "D:/Minor Project/processed_data"
os.makedirs(PROC_DIR, exist_ok=True)

RAW_FILES = {
    "2010-11": os.path.join(RAW_DIR, "2010-2011 Solar home electricity data.csv"),
    "2011-12": os.path.join(RAW_DIR, "2011-2012 Solar home electricity data v2.csv"),
    "2012-13": os.path.join(RAW_DIR, "2012-2013 Solar home electricity data v2.csv"),
}

# 48 half-hour column names as they appear in the CSV
TIME_COLS = (
    [f"{h}:{m}" for h in range(0, 24) for m in ["30", "00"]][1:]
)
TIME_COLS = TIME_COLS + ["0:00"]   # last slot
print("TIME_COLS count:", len(TIME_COLS))


TIME_COLS count: 48


## 2. Load & merge all three year CSVs

In [6]:
def load_year(path, year_label):
    """
    Each CSV has a title string in row 0 and the real header in row 1.
    We skip row 0 and rename the first 5 descriptor columns.
    """
    raw = pd.read_csv(path, header=1, low_memory=False)
    raw = raw.iloc[1:].reset_index(drop=True)   # drop title row

    rename = {
        raw.columns[0]: "customer_id",
        raw.columns[1]: "generator_kwp",
        raw.columns[2]: "postcode",
        raw.columns[3]: "category",
        raw.columns[4]: "date",
    }
    raw.rename(columns=rename, inplace=True)

    # rename the 48 half-hour columns
    for i, tc in enumerate(TIME_COLS):
        if 5 + i < len(raw.columns):
            raw.rename(columns={raw.columns[5 + i]: tc}, inplace=True)

    # optional row-quality column (col 54)
    if len(raw.columns) > 53:
        raw.rename(columns={raw.columns[53]: "row_quality"}, inplace=True)
    else:
        raw["row_quality"] = ""

    raw["year_file"] = year_label
    return raw

frames = []
for label, path in RAW_FILES.items():
    df_yr = load_year(path, label)
    frames.append(df_yr)
    print(f"Loaded {label}: {len(df_yr):,} rows")

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows: {len(df_raw):,}")
df_raw.head(3)


Loaded 2010-11: 269,734 rows
Loaded 2011-12: 270,303 rows
Loaded 2012-13: 268,556 rows

Total rows: 808,593


,customer_id,generator_kwp,postcode,category,date,0:00,1:00,1:00,2:00,2:00,...,20:00,21:00,21:00,22:00,22:00,23:00,23:00,0:00,row_quality,year_file
0,1,3.78,2076,CL,1-Jul-10,1.250,1.244,1.256,0.744,0.019,...,0.000,0.000,0.000,0.000,0.000,0.000,0.00,1.075,,2010-11
1,1,3.78,2076,GG,1-Jul-10,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.000,,2010-11
2,1,3.78,2076,GC,2-Jul-10,0.116,0.346,0.122,0.079,0.120,...,1.012,0.817,0.526,0.335,0.402,0.142,0.12,0.111,,2010-11


## 3. Parse dates — three different formats across years

In [7]:
def parse_dates(series):
    """
    2010-11 uses  1-Jul-10,  2011-12+ uses  1/07/2011
    Try both; coerce anything leftover.
    """
    parsed = pd.to_datetime(series.astype(str).str.strip(), dayfirst=True, errors="coerce")
    mask   = parsed.isna()
    if mask.any():
        fallback = pd.to_datetime(series[mask], format="%d-%b-%y", errors="coerce")
        parsed[mask] = fallback
    return parsed

df_raw["date"] = parse_dates(df_raw["date"])
print("Date range:", df_raw["date"].min().date(), "→", df_raw["date"].max().date())
print("Unparsed dates:", df_raw["date"].isna().sum())


Date range: 2010-07-01 → 2013-06-30
Unparsed dates: 0


## 4. Cast types & basic cleaning

In [9]:
# Numeric casts
df_raw["customer_id"]   = pd.to_numeric(df_raw["customer_id"],  errors="coerce").astype("Int64")
df_raw["generator_kwp"] = pd.to_numeric(df_raw["generator_kwp"], errors="coerce")

# Fix duplicate column names before casting (caused by bad TIME_COLS)
df_raw.columns = pd.io.parsers.base_parser.ParserBase({'usecols': None})._maybe_dedup_names(df_raw.columns) \
    if hasattr(pd.io.parsers.base_parser.ParserBase({'usecols': None}), '_maybe_dedup_names') \
    else df_raw.columns

# Safer approach: only cast columns that are actually Series (not duplicate DataFrames)
for tc in TIME_COLS:
    if tc in df_raw.columns:
        col = df_raw[tc]
        if isinstance(col, pd.DataFrame):
            # duplicate column — keep only the first occurrence
            df_raw = df_raw.loc[:, ~df_raw.columns.duplicated()]
        df_raw[tc] = pd.to_numeric(df_raw[tc], errors="coerce").fillna(0.0)

# Flag non-actual rows
df_raw["is_non_actual"] = (
    df_raw["row_quality"].astype(str).str.strip().str.upper() == "NA"
)

# Keep only valid categories
valid_cats = {"GC", "CL", "GG"}
before = len(df_raw)
df_raw = df_raw[df_raw["category"].isin(valid_cats)].copy()
df_raw = df_raw.dropna(subset=["date"]).reset_index(drop=True)

print(f"Rows after cleaning: {len(df_raw):,}  (dropped {before - len(df_raw):,})")
print(f"Non-actual rows flagged: {df_raw['is_non_actual'].sum():,}")
df_raw[["customer_id","date","category","generator_kwp","is_non_actual"]].head(5)

Rows after cleaning: 808,593  (dropped 0)
Non-actual rows flagged: 0


,customer_id,date,category,generator_kwp,is_non_actual
0,1,2010-07-01,CL,3.78,False
1,1,2010-07-01,GG,3.78,False
2,1,2010-07-02,GC,3.78,False
3,1,2010-07-02,CL,3.78,False
4,1,2010-07-02,GG,3.78,False


## 5. Reshape: wide → long → pivot

In [10]:
present_tc = [c for c in TIME_COLS if c in df_raw.columns]
id_cols    = ["customer_id","generator_kwp","postcode","category","date","year_file","is_non_actual"]

# Melt the 48 time columns into rows
long = df_raw[id_cols + present_tc].melt(
    id_vars=id_cols,
    value_vars=present_tc,
    var_name="time_str",
    value_name="kwh",
)

# Build full timestamp
long["time_str"] = long["time_str"].str.zfill(5)          # "0:30" → "00:30"
long["timestamp"] = pd.to_datetime(
    long["date"].astype(str) + " " + long["time_str"], errors="coerce"
)
long.drop(columns=["time_str"], inplace=True)
long["kwh"] = pd.to_numeric(long["kwh"], errors="coerce").fillna(0.0)

print(f"Long format: {long.shape}")
long.head(3)


Long format: (20214825, 9)


,customer_id,generator_kwp,postcode,category,date,year_file,is_non_actual,kwh,timestamp
0,1,3.78,2076,CL,2010-07-01,2010-11,False,1.250,2010-07-01
1,1,3.78,2076,GG,2010-07-01,2010-11,False,0.000,2010-07-01
2,1,3.78,2076,GC,2010-07-02,2010-11,False,0.116,2010-07-02


In [11]:
# Pivot so GC, CL, GG are separate columns per timestamp
pivot = long.pivot_table(
    index=["customer_id","generator_kwp","postcode","date","timestamp","year_file"],
    columns="category",
    values="kwh",
    aggfunc="first",
).reset_index()
pivot.columns.name = None

for cat in ["GC","CL","GG"]:
    if cat not in pivot.columns:
        pivot[cat] = 0.0

pivot = pivot.sort_values(["customer_id","timestamp"]).reset_index(drop=True)
print(f"Pivot shape: {pivot.shape}")
pivot.head(3)


Pivot shape: (7889256, 9)


,customer_id,generator_kwp,postcode,date,timestamp,year_file,CL,GC,GG
0,1,3.78,2076,2010-07-01,2010-07-01 00:00:00,2010-11,1.250,NaN,0.0
1,1,3.78,2076,2010-07-01,2010-07-01 01:00:00,2010-11,1.244,NaN,0.0
2,1,3.78,2076,2010-07-01,2010-07-01 02:00:00,2010-11,0.744,NaN,0.0


## 6. Feature engineering

In [12]:
df = pivot.copy()
df = df.sort_values(["customer_id","timestamp"]).reset_index(drop=True)

# ── Time features ──────────────────────────────────────────────────────────
ts           = pd.to_datetime(df["timestamp"])
df["hour_slot"]  = ts.dt.hour * 2 + (ts.dt.minute == 30).astype(int)  # 0-47
df["dayofweek"]  = ts.dt.dayofweek
df["month"]      = ts.dt.month
df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
df["season"]     = ts.dt.month.map(
    {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
)

# ── Lag features (per customer) ────────────────────────────────────────────
grp = df.groupby("customer_id")["GC"]
for lag in [1, 2, 48, 336]:
    df[f"GC_lag_{lag}"] = grp.shift(lag)

# ── Rolling means ──────────────────────────────────────────────────────────
for days in [3, 7]:
    df[f"GC_roll_{days}d"] = (
        grp.shift(1).rolling(days * 48, min_periods=days * 24).mean()
           .reset_index(level=0, drop=True)
    )

# ── Solar offset: yesterday's total GG ────────────────────────────────────
daily_gg = (
    df.groupby(["customer_id","date"])["GG"].sum().reset_index()
      .rename(columns={"GG":"prev_gg","date":"date"})
)
daily_gg["date"] = pd.to_datetime(daily_gg["date"])
daily_gg["date_next"] = daily_gg["date"] + pd.Timedelta(days=1)
df["date"] = pd.to_datetime(df["date"])
df = df.merge(
    daily_gg[["customer_id","date_next","prev_gg"]],
    left_on=["customer_id","date"], right_on=["customer_id","date_next"], how="left"
).drop(columns=["date_next"])
df["prev_gg"] = df["prev_gg"].fillna(0.0)

# ── Capacity-normalised GC ─────────────────────────────────────────────────
df["GC_per_kwp"] = df["GC"] / df["generator_kwp"].replace(0, np.nan)

# Drop rows with lag NaNs (start of each customer's series)
lag_cols = [f"GC_lag_{l}" for l in [1,2,48,336]]
before = len(df)
df = df.dropna(subset=lag_cols).reset_index(drop=True)
print(f"Dropped {before-len(df):,} warmup rows. Final shape: {df.shape}")
df[["customer_id","timestamp","GC","GC_lag_48","GC_roll_7d","prev_gg","hour_slot"]].head(5)


Dropped 100,824 warmup rows. Final shape: (7788432, 22)


,customer_id,timestamp,GC,GC_lag_48,GC_roll_7d,prev_gg,hour_slot
0,1,2010-07-16 00:00:00,0.105,0.099,0.581301,6.296,0
1,1,2010-07-16 01:00:00,0.130,0.127,0.581268,6.296,2
2,1,2010-07-16 02:00:00,0.375,0.096,0.580625,6.296,4
3,1,2010-07-16 03:00:00,0.102,0.082,0.581506,6.296,6
4,1,2010-07-16 04:00:00,0.083,0.105,0.581491,6.296,8


## 7. Save processed dataset

In [13]:
out_path = os.path.join(PROC_DIR, "final_dataset.csv")
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(df.dtypes)


Saved → D:/Minor Project/processed_data\final_dataset.csv
customer_id               Int64
generator_kwp           float64
postcode                  int64
date             datetime64[us]
timestamp        datetime64[us]
year_file                   str
CL                      float64
GC                      float64
GG                      float64
hour_slot                 int64
dayofweek                 int32
month                     int32
is_weekend                int64
season                    int64
GC_lag_1                float64
GC_lag_2                float64
GC_lag_48               float64
GC_lag_336              float64
GC_roll_3d              float64
GC_roll_7d              float64
prev_gg                 float64
GC_per_kwp              float64
dtype: object
